# NeuraCommand Quick Start

This notebook demonstrates the basic usage of NeuraCommand - a lightweight neural network library designed for edge devices.

## Features
- Simple Keras-like API
- Pure NumPy (minimal dependencies)
- Optimized for Raspberry Pi, Jetson Nano, IoT devices
- Optional GPU support via CuPy

In [ ]:
# Install if needed (uncomment)
# !pip install -e ..

In [ ]:
import numpy as np
import sys
sys.path.insert(0, '../src')

from neura_command import (
    Sequential, Dense, ReLU, Sigmoid, Softmax, Dropout,
    Adam, SGD, MSELoss, CrossEntropyLoss,
    EarlyStopping,
    __version__
)

print(f"NeuraCommand v{__version__}")

## 1. Create a Simple Model

Let's create a neural network with 2 hidden layers.

In [ ]:
# Create a simple model
model = Sequential(
    layers=[
        Dense(4, 16),      # 4 inputs -> 16 hidden units
        ReLU(),
        Dense(16, 8),      # 16 -> 8 hidden units
        ReLU(),
        Dense(8, 1),       # 8 -> 1 output
        Sigmoid(),         # Output between 0 and 1
    ],
    optimizer=Adam(learning_rate=0.01),
    loss=MSELoss(),
)

print(model.summary())

## 2. Generate Sample Data

We'll create a simple XOR-like problem.

In [ ]:
np.random.seed(42)

# Generate synthetic data (noisy XOR)
n_samples = 500
X = np.random.randn(n_samples, 4).astype(np.float32)

# Target: XOR of signs of first two features
y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(np.float32).reshape(-1, 1)

# Split into train/test
split = int(0.8 * n_samples)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Input shape: {X_train.shape}")
print(f"Output shape: {y_train.shape}")

## 3. Train the Model

In [ ]:
# Train with validation and early stopping
history = model.train(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[EarlyStopping(patience=5, min_delta=0.001)],
    verbose=True,
)

## 4. Evaluate the Model

In [ ]:
# Make predictions
predictions = model.predict(X_test)

# Convert to binary predictions
binary_preds = (predictions > 0.5).astype(int)

# Calculate accuracy
accuracy = np.mean(binary_preds == y_test) * 100
print(f"Test Accuracy: {accuracy:.1f}%")

# Show some predictions
print("\nSample predictions:")
for i in range(5):
    print(f"  Pred: {predictions[i, 0]:.3f} -> {binary_preds[i, 0]}, True: {int(y_test[i, 0])}")

## 5. Save and Load the Model

In [ ]:
import tempfile
import os

# Save the model
with tempfile.TemporaryDirectory() as tmpdir:
    model_path = os.path.join(tmpdir, 'my_model')
    
    # Save
    model.save(model_path)
    print(f"Model saved to {model_path}")
    print(f"Files created: {os.listdir(tmpdir)}")
    
    # Load
    loaded_model = Sequential.load(model_path)
    print(f"\nLoaded model:")
    print(loaded_model.summary())
    
    # Verify predictions match
    loaded_preds = loaded_model.predict(X_test)
    print(f"\nPredictions match: {np.allclose(predictions, loaded_preds)}")

## 6. Using Different Optimizers

In [ ]:
from neura_command import SGD, RMSprop, AdaGrad

optimizers = {
    'Adam': Adam(learning_rate=0.01),
    'SGD': SGD(learning_rate=0.1, momentum=0.9),
    'RMSprop': RMSprop(learning_rate=0.01),
    'AdaGrad': AdaGrad(learning_rate=0.1),
}

for name, opt in optimizers.items():
    model = Sequential(
        [Dense(4, 16), ReLU(), Dense(16, 1), Sigmoid()],
        optimizer=opt,
        loss=MSELoss(),
    )
    model.train(X_train, y_train, epochs=20, batch_size=32, verbose=False)
    pred = (model.predict(X_test) > 0.5).astype(int)
    acc = np.mean(pred == y_test) * 100
    print(f"{name:10s}: {acc:.1f}% accuracy")

## Summary

NeuraCommand provides:

- **Simple API**: Keras-like `Sequential` model
- **Layers**: Dense, Dropout, BatchNorm, ReLU, Sigmoid, Softmax, etc.
- **Optimizers**: Adam, SGD (with momentum), RMSprop, AdaGrad
- **Callbacks**: EarlyStopping, LearningRateScheduler
- **Serialization**: Safe JSON + NPZ format (no pickle)
- **Edge-ready**: Quantization and lightweight inference runtime

Next notebooks:
- `02_linear_regression.ipynb` - Regression example
- `03_logical_operations.ipynb` - AND/OR/XOR gates
- `04_digit_recognition.ipynb` - MNIST classification
- `05_model_serialization.ipynb` - Save/load deep dive
- `06_quantization_edge.ipynb` - Edge deployment